# Quality Governance Setup

## Purpose

In this notebook, I create the persistent Unity Catalog objects used to manage
data-quality rules for the project.

I store quality rules centrally rather than embedding them permanently inside
individual Lakeflow pipeline files.

This allows quality rules to be:

- updated independently from transformation code
- reused by multiple pipeline datasets
- enabled or disabled without changing pipeline logic
- version-controlled through metadata
- monitored and governed centrally

### Architecture

`health_insurance.governance.quality_rules`

will act as the production source of truth for Lakeflow expectation
definitions.

The profiling notebooks in `04-data-quality` determine and approve the rules,
while the Lakeflow pipeline retrieves and enforces the active rules.

In [0]:
%sql
-- ensuring that the project catalog exists.

CREATE CATALOG IF NOT EXISTS health_insurance;

In [0]:
%sql
-- creating a governance schema for reusable
-- data-platform metadata and quality configuration.

CREATE SCHEMA IF NOT EXISTS health_insurance.governance
COMMENT 'Governance metadata, quality rules, and reusable platform configuration for the health insurance lakehouse.';

In [0]:
%sql
-- creating the central repository used to store
-- approved data-quality rules for Lakeflow pipelines.

CREATE TABLE IF NOT EXISTS health_insurance.governance.quality_rules (
    dataset STRING NOT NULL,
    rule_name STRING NOT NULL,
    constraint STRING NOT NULL,
    severity STRING NOT NULL,
    is_active BOOLEAN NOT NULL,
    description STRING,
    owner STRING,
    version INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Central repository of reusable Lakeflow data-quality expectation rules.';

In [0]:
%sql
ALTER TABLE health_insurance.governance.quality_rules
SET TBLPROPERTIES (
    'quality' = 'configuration',
    'project.layer' = 'governance',
    'purpose' = 'lakeflow_expectation_rules'
);

In [0]:
%sql
DESCRIBE TABLE EXTENDED
health_insurance.governance.quality_rules;

In [0]:
%sql
-- adding constraints to the quality rules table

ALTER TABLE health_insurance.governance.quality_rules
ADD CONSTRAINT valid_quality_rule_severity
CHECK (severity IN ('WARN', 'DROP', 'FAIL'));

ALTER TABLE health_insurance.governance.quality_rules
ADD CONSTRAINT valid_quality_rule_version
CHECK (version IS NULL OR version >= 1);

In [0]:
%sql
ALTER TABLE health_insurance.governance.quality_rules
ADD COLUMNS (
    source_notebook STRING
);